In [168]:
%cd ../src

/home/parthgandhi/argus/argus/src


In [ ]:
import polars as pl

from config.brokers import KiteConfig

In [170]:
_tbl_name = KiteConfig.get_db_tbl(
    data_type=KiteConfig.TYP_STOCKS, frequency="day", failed_tbl=False
)

In [171]:
data = pl.read_database_uri(
    query=f"SELECT * from {_tbl_name}", uri=KiteConfig.DB_CONN, engine="connectorx"
)

In [ ]:
class BasePopConfig:
    ADDV = 50
    IPO_ADDV = 2_50_00_000
    STOCKS_ADDV = 5_00_00_000

In [177]:
BasePopConfig.ADDV = 10

In [178]:
BasePopConfig.ADDV

10

In [173]:
_max_timestamp = (
    data.lazy()
    .with_columns(pl.col("timestamp").cast(pl.Date()))
    .select("timestamp")
    .max()
    .collect()
    .item(0, 0)
)

df = (
    data.lazy()
    .with_columns(pl.col("timestamp").cast(pl.Date()))
    .with_columns(
        # Dollar Volume
        [
            (
                (pl.col("close") * pl.col("volume")).rolling_mean(
                    window_size=BasePopConfig.ADDV
                )
            )
            .round(2)
            .over(
                partition_by="symbol",
                order_by="timestamp",
                descending=False,
            )
            .alias(f"dollar_volume_sma_{BasePopConfig.ADDV}")
        ]
    )
)

In [174]:
ipo_stocks_list = (
    df.filter(
        pl.col(f"dollar_volume_sma_{BasePopConfig.ADDV}").is_null()
        & (pl.col("timestamp") == _max_timestamp)
    )
    .sort("symbol")
    .collect()
    .get_column("symbol")
    .to_list()
)

In [175]:
ipo_stocks_df = (
    df.filter(pl.col("symbol").is_in(ipo_stocks_list))
    .group_by("symbol")
    .agg(
        pl.col("timestamp").max().alias("timestamp"),
        (pl.col("close") * pl.col("volume")).mean().round(2).alias("avg_dollar_volume"),
        pl.col("timestamp").len().alias("days_since_ipo"),
    )
    .filter(pl.col("avg_dollar_volume") >= BasePopConfig.IPO_ADDV)
    .sort("symbol")
    .collect()
)

In [176]:
base_pop_df = df.filter(
    (pl.col("timestamp") == _max_timestamp)
    & (pl.col(f"dollar_volume_sma_{BasePopConfig.ADDV}") >= BasePopConfig.STOCKS_ADDV)
).collect()